In [1]:
import random

# 1. Clase Base del Problema
class Problem:
    def __init__(self, initial, goal=None):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def value(self, state):
        """Heurística"""
        raise NotImplementedError

# 2. Clase Nodo
class Node:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action

    def expand(self, problem):
        return [self.child_node(problem, action)
                for action in problem.actions(self.state)]

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        return Node(next_state, self, action)

    def path(self):
        node, path_back = self, []
        while node:
            path_back.append(node.state)
            node = node.parent
        return path_back[::-1]

# 3.Hill Climbing
def hill_climbing(problem):
    """Busqueda que selecciona el vecino con el valor de evaluación mas alto."""
    current = Node(problem.initial)
    while True:
        neighbors = current.expand(problem)
        if not neighbors:
            break
            
        # Selecciona el vecino con el valor max
        neighbor = max(neighbors, key=lambda node: problem.value(node.state))
        
        # Si el mejor vecino no mejora al estado actual, alcanzamos un max local
        if problem.value(neighbor.state) <= problem.value(current.state):
            break
            
        current = neighbor
    return current

In [2]:
class HillClimbingGraphProblem(Problem):
    def __init__(self, initial, goal, graph, heuristics):
        super().__init__(initial, goal)
        self.graph = graph
        self.heuristics = heuristics  # Estimacion hacia la meta 

    def actions(self, state):
        return list(self.graph.get(state, {}).keys())

    def result(self, state, action):
        return action

    def value(self, state):
        
        return -self.heuristics.get(state, float('inf'))

# Grafo de prueba con heuristicas 
metro_hc = {
    'Cuatro Caminos': {'Tacuba': 1},
    'Tacuba': {'Cuatro Caminos': 1, 'Hidalgo': 1},
    'Hidalgo': {'Tacuba': 1, 'Bellas Artes': 1, 'Balderas': 1},
    'Bellas Artes': {'Hidalgo': 1, 'Pino Suarez': 1, 'Salto del Agua': 1},
    'Pino Suarez': {'Bellas Artes': 1, 'Pantitlan': 1},
    'Balderas': {'Hidalgo': 1, 'Salto del Agua': 1},
    'Salto del Agua': {'Balderas': 1, 'Bellas Artes': 1, 'Pino Suarez': 1},
    'Pantitlan': {}
}

# Distancia heuristica estimada al destino 'Pantitlan'
heuristica_pantitlan = {
    'Cuatro Caminos': 15,
    'Tacuba': 12,
    'Hidalgo': 8,
    'Balderas': 9,
    'Salto del Agua': 7,
    'Bellas Artes': 5,
    'Pino Suarez': 2,
    'Pantitlan': 0
}

# Ejecucion
problema_hc = HillClimbingGraphProblem('Cuatro Caminos', 'Pantitlan', metro_hc, heuristica_pantitlan)
solucion = hill_climbing(problema_hc)

print("RESULTADO HILL CLIMBING")
print("Camino recorrido:", " -> ".join(solucion.path()))
print("Estado final alcanzado:", solucion.state)
print("Valor de evaluación final:", problema_hc.value(solucion.state))

RESULTADO HILL CLIMBING
Camino recorrido: Cuatro Caminos -> Tacuba -> Hidalgo -> Bellas Artes -> Pino Suarez -> Pantitlan
Estado final alcanzado: Pantitlan
Valor de evaluación final: 0
